# GPTQ và AWQ tự cài trên GPT mini

Round-to-nearest (RTN) làm tròn từng trọng số độc lập, không biết trọng số nào quan trọng. Hai phương pháp PTQ phổ biến nhất
cho LLM dùng thêm một ít **dữ liệu hiệu chỉnh** (calibration) để làm tốt hơn: **GPTQ** sửa sai số làm tròn của mỗi cột bằng
cách cập nhật các cột chưa quantize theo Hessian của lớp, **AWQ** phóng to các kênh input có activation lớn trước khi
quantize. Bạn tự cài cả hai cho từng `nn.Linear` (mục 5 phần lý thuyết), áp lên GPT mini của module 05 và so với RTN ở cùng
số bit.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`. Cell `rtn_quantize` cho sẵn.
- Cell gắn tag `explore` nạp checkpoint, hiệu chỉnh, đánh giá; test bỏ qua chúng.
- Checkpoint: như bài Intermediate 01 (`data/raw/gpt-mini-vi/gpt_mini_vi.pt` hoặc biến `GPT_MINI_CKPT`, chưa có thì tự train
  bản dự phòng).
- Chấm: `uv run pytest tracks/ai-engineer/modules/10-quantization/exercises/intermediate/02-gptq-awq -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt (tải một lần bằng env gốc), chỗ để checkpoint
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"
CKPT = Path(os.environ.get("GPT_MINI_CKPT", ROOT / "data/raw/gpt-mini-vi/gpt_mini_vi.pt"))
FALLBACK = Path(os.environ.get("GPT_MINI_FALLBACK", ROOT / "data/raw/gpt-mini-vi/fallback_small.pt"))
print("checkpoint:", CKPT, "có" if CKPT.exists() else "chưa có")

In [ ]:
import copy
import math

import torch
import torch.nn as nn

## 0. Cho sẵn: RTN theo nhóm

`rtn_quantize(W, bits, group_size, symmetric=True)`: fake quantization theo nhóm `group_size` cột liền nhau của một hàng,
giống bài Basic 02 và Intermediate 01. `quant_params` và `quant_column` là hai mảnh nhỏ mà GPTQ cần: tham số lượng tử của
một khối cột, và lượng tử hoá một cột với tham số đó.

In [ ]:
def quant_params(block: torch.Tensor, bits: int, symmetric: bool):
    """Scale (và zero) cho mỗi hàng của `block` shape (out, k). Trả (scale (out,), zero (out,) hoặc None)."""
    if symmetric:
        qmax = 2 ** (bits - 1) - 1
        scale = block.abs().amax(dim=1) / qmax
        return torch.where(scale == 0, torch.ones_like(scale), scale), None
    qmax = 2**bits - 1
    lo = block.amin(dim=1).clamp(max=0)
    hi = block.amax(dim=1).clamp(min=0)
    scale = (hi - lo) / qmax
    scale = torch.where(scale == 0, torch.ones_like(scale), scale)
    return scale, torch.clamp(torch.round(-lo / scale), 0, qmax)


def quant_column(col: torch.Tensor, scale: torch.Tensor, zero, bits: int) -> torch.Tensor:
    """Quantize rồi dequantize một cột (out,) với tham số của `quant_params`."""
    if zero is None:
        qmax = 2 ** (bits - 1) - 1
        return torch.clamp(torch.round(col / scale), -qmax, qmax) * scale
    qmax = 2**bits - 1
    return (torch.clamp(torch.round(col / scale) + zero, 0, qmax) - zero) * scale


def rtn_quantize(W: torch.Tensor, bits: int, group_size: int, symmetric: bool = True) -> torch.Tensor:
    out = torch.empty_like(W)
    for start in range(0, W.shape[1], group_size):
        block = W[:, start:start + group_size]
        scale, zero = quant_params(block, bits, symmetric)
        z = None if zero is None else zero[:, None]
        out[:, start:start + group_size] = quant_column(block, scale[:, None], z, bits)
    return out

## 1. Dữ liệu hiệu chỉnh: input của từng lớp

- `capture_inputs(model, batches, names)`: chạy model (chế độ eval, không gradient) trên từng batch token trong `batches`,
  ghi lại **input** của mỗi module có tên trong `names` bằng forward hook. Trả `dict` tên → tensor `float32` trên CPU shape
  `(tổng số token, in_features)`, các batch nối theo thứ tự. Gỡ hết hook trước khi trả.
- `hessian(X, damp=0.01)`: $$H = \frac{2}{N} X^\top X + \lambda I$$ với $$\lambda$$ = `damp` × trung bình đường chéo của
  $$\frac{2}{N} X^\top X$$. Trả `float64`. Đây là Hessian của sai số đầu ra bình phương $$\lVert XW^\top - X\hat W^\top\rVert^2 / N$$
  theo mỗi hàng của $$W$$; số hạng $$\lambda I$$ (dampening) giữ $$H$$ khả nghịch khi có kênh input gần như luôn bằng 0.

In [ ]:
@torch.no_grad()
def capture_inputs(model: nn.Module, batches, names: list[str]) -> dict[str, torch.Tensor]:
    # TODO: register_forward_hook trên model.get_submodule(name); hook nhận (module, inputs, output)
    raise NotImplementedError


def hessian(X: torch.Tensor, damp: float = 0.01) -> torch.Tensor:
    # TODO
    raise NotImplementedError

## 2. GPTQ

`gptq_quantize(W, H, bits, group_size, symmetric=True)`: `W` shape `(out, in)`, `H` shape `(in, in)` đã có dampening (không
thêm gì nữa). Duyệt cột $$j = 0, 1, \dots$$ **theo thứ tự**:

1. Ở cột đầu mỗi nhóm ($$j$$ chia hết cho `group_size`), tính tham số lượng tử của nhóm bằng `quant_params` trên các cột
   `j .. j + group_size` **hiện tại** của `W` (đã được các bước trước cập nhật).
2. Quantize cột $$j$$ bằng `quant_column`, ghi vào kết quả.
3. Bù sai số vào các cột chưa quantize: với $$U$$ là nhân tử Cholesky **trên** của $$H^{-1}$$ ($$H^{-1} = U^\top U$$),
   $$e = (w_j - q_j) / U_{jj}$$, rồi $$W_{:, j+1:} \leftarrow W_{:, j+1:} - e\, U_{j, j+1:}$$.

Tính trong `float64`, trả tensor đã dequantize cùng dtype với `W`, không sửa `W` đầu vào. Gợi ý: `torch.linalg.cholesky`,
`torch.cholesky_inverse`, `torch.linalg.cholesky(..., upper=True)`.

In [ ]:
def gptq_quantize(W: torch.Tensor, H: torch.Tensor, bits: int, group_size: int, symmetric: bool = True) -> torch.Tensor:
    # TODO
    raise NotImplementedError

## 3. AWQ

Nhân cột $$k$$ của $$W$$ với $$s_k > 0$$ và chia input kênh $$k$$ cho $$s_k$$ không đổi đầu ra của lớp, nhưng làm các trọng số
gặp activation lớn được làm tròn tương đối chính xác hơn. Với activation đã chia sẵn, trọng số hiệu dụng là
$$Q(W \cdot s) / s$$.

- `awq_scales(x_absmean, alpha)`: $$s = m^{\alpha}$$ với $$m$$ là trung bình $$|x|$$ của từng kênh input (kẹp dưới ở `1e-5`),
  rồi chuẩn hoá $$s \leftarrow s / \sqrt{\max s \cdot \min s}$$.
- `awq_quantize(W, X, bits, group_size, n_grid=20, symmetric=True)`: thử $$\alpha = i / n_{grid}$$ với $$i = 0 .. n_{grid} - 1$$;
  với mỗi $$\alpha$$, trọng số hiệu dụng `rtn_quantize(W * s, ...) / s`, sai số là trung bình bình phương của
  $$XW^\top - X\hat W^\top$$. Trả `(trọng số hiệu dụng tốt nhất, alpha tốt nhất)`; hoà thì giữ $$\alpha$$ nhỏ hơn.

In [ ]:
def awq_scales(x_absmean: torch.Tensor, alpha: float) -> torch.Tensor:
    # TODO
    raise NotImplementedError


def awq_quantize(W: torch.Tensor, X: torch.Tensor, bits: int, group_size: int, n_grid: int = 20,
                 symmetric: bool = True) -> tuple[torch.Tensor, float]:
    # TODO
    raise NotImplementedError

## 4. Áp lên cả model

`apply_layerwise(model, inputs, quantize_fn)`: trả bản sao của model (gốc không đổi) trong đó với mỗi tên `n` trong `inputs`,
trọng số của module `n` được thay bằng `quantize_fn(n, W, inputs[n])` (`W` là trọng số của bản sao). Các module khác giữ
nguyên.

In [ ]:
def apply_layerwise(model: nn.Module, inputs: dict[str, torch.Tensor], quantize_fn) -> nn.Module:
    # TODO
    raise NotImplementedError

## 5. Thí nghiệm

Các cell dưới đây không được chấm. Trên CPU, mỗi cấu hình mất khoảng 1-3 phút (GPTQ vài giây tới vài chục giây, AWQ thử 20 giá
trị $$\alpha$$ cho mỗi lớp nên lâu nhất), cả mục khoảng 10 phút với checkpoint cấu hình PC.

In [ ]:
# explore: nạp lớp GPT (Basic 01) và các hàm dữ liệu, đánh giá (Intermediate 01) của module 05: bài của bạn nếu đã làm, không thì lời giải
import importlib.util

from testkit import Exercise

M05 = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"


def first_working(loaders, probe):
    for load in loaders:
        module = load()
        try:
            probe(module)
            return module
        except NotImplementedError:
            continue
    raise RuntimeError("không nạp được code module 05")


def py_loader(path):
    def load():
        spec = importlib.util.spec_from_file_location("gpt_mini", path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["gpt_mini"] = module
        spec.loader.exec_module(module)
        return module
    return load


def nb_loader(target):
    def load():
        ex05 = Exercise(str(M05 / "exercises/intermediate/01-train-gpt-wikipedia/tests/test_train_gpt.py"))
        ex05.target = target
        return ex05.notebook("train_gpt.ipynb")
    return load


gpt_mini = first_working(
    [py_loader(M05 / "exercises/basic/01-gpt-mini/starter/gpt.py"), py_loader(M05 / "solutions/basic/01-gpt-mini/gpt.py")],
    lambda m: m.GPT(m.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(torch.zeros(1, 2, dtype=torch.long)),
)
m05 = first_working(
    [nb_loader(M05 / "exercises/intermediate/01-train-gpt-wikipedia/starter"),
     nb_loader(M05 / "solutions/intermediate/01-train-gpt-wikipedia")],
    lambda m: (m.sliding_windows(10, 4, 2), m.split_articles(["a", "b", "c"], 1, 1, 0), m.nll_to_metrics(1.0, 1, 1),
               m.lr_at(0, 1e-3, 10, 100, 1e-4)),
)
print("GPT:", gpt_mini.__file__)
print("hàm đánh giá:", m05.__file__)

In [ ]:
# explore: nạp checkpoint gpt-mini-v1 của bạn; chưa có thì train một GPT nhỏ dự phòng (CPU khoảng 4 phút, lưu lại để dùng tiếp)
import dataclasses
import time

from tokenizers import Tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
articles = m05.long_articles(SHARD, 2000)
train_articles, val_articles = m05.split_articles(articles, 1000, 100, seed=0)


def load_gpt_mini(path):
    raw = torch.load(path, map_location="cpu", weights_only=True)
    if raw.get("format") != "gpt-mini-v1":
        raise ValueError(f"{path} không phải định dạng gpt-mini-v1")
    model = gpt_mini.GPT(gpt_mini.GPTConfig(**raw["config"]))
    model.load_state_dict(raw["state_dict"])
    return model.eval(), Tokenizer.from_str(raw["tokenizer"])


def train_fallback(path, steps=600):
    tok = m05.train_bpe("\n\n".join(train_articles), 8000)
    ids = torch.tensor(tok.encode("\n\n".join(train_articles)).ids, dtype=torch.long)
    torch.manual_seed(0)
    config = gpt_mini.GPTConfig(vocab_size=8000, block_size=128, n_layer=4, n_head=4, n_embd=128)
    model = gpt_mini.GPT(config)
    opt = m05.configure_optimizer(model, 1e-3, 0.1)
    g = torch.Generator().manual_seed(0)
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = m05.lr_at(step, 1e-3, 100, steps, 1e-4)
        x, y = m05.get_batch(ids, 32, 128, g)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"format": "gpt-mini-v1", "config": dataclasses.asdict(config),
                "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "tokenizer": tok.to_str(), "step": steps, "metrics": {}}, path)


source = CKPT if CKPT.exists() else FALLBACK
if not source.exists():
    print("chưa có checkpoint module 05: train GPT nhỏ dự phòng...")
    start = time.perf_counter()
    train_fallback(FALLBACK)
    print(f"xong sau {time.perf_counter() - start:.0f} s, lưu ở {FALLBACK}")
model, tok = load_gpt_mini(source)
BLOCK = model.config.block_size
print(source.name, model.config, f"{model.num_params():,} tham số")

In [ ]:
# explore: validation của module 04/05; CPU chỉ lấy 20 bài đầu cho nhanh, GPU lấy đủ 100 bài
N_VAL = 100 if DEVICE == "cuda" else 20
val_text = "\n\n".join(val_articles[:N_VAL])
val_ids = torch.tensor(tok.encode(val_text).ids, dtype=torch.long)


def val_bpc(m) -> float:
    total, n = m05.evaluate_nll(m.to(DEVICE), val_ids, BLOCK, BLOCK // 2, device=DEVICE)
    return m05.nll_to_metrics(total, n, len(val_text))["bpc"]


start = time.perf_counter()
BASE_BPC = val_bpc(model)
print(f"{N_VAL} bài, {len(val_text):,} ký tự, {len(val_ids):,} token | float32 bpc {BASE_BPC:.4f}"
      f" ({time.perf_counter() - start:.1f} s mỗi lần đánh giá trên {DEVICE})")

In [ ]:
# explore: dữ liệu hiệu chỉnh: 32 đoạn BLOCK token ngẫu nhiên từ tập train của module 04/05 (không lấy validation)
train_ids = torch.tensor(tok.encode("\n\n".join(train_articles[:200])).ids, dtype=torch.long)
g = torch.Generator().manual_seed(0)
starts = torch.randint(len(train_ids) - BLOCK, (32,), generator=g)
calib = torch.stack([train_ids[s:s + BLOCK] for s in starts])
batches = [calib[i:i + 8].to(DEVICE) for i in range(0, len(calib), 8)]
names = [n for n, m in model.named_modules() if isinstance(m, nn.Linear) and m.weight is not model.tok_emb.weight]
start = time.perf_counter()
inputs = capture_inputs(model.to(DEVICE), batches, names)
print(f"{len(names)} lớp, mỗi lớp {inputs[names[0]].shape[0]:,} vector input ({time.perf_counter() - start:.1f} s)")

In [ ]:
# explore: activation có ngoại lai theo kênh không? max / median của trung bình |x| theo kênh input
for n in names:
    m = inputs[n].abs().mean(dim=0)
    print(f"{n:22s} in={m.numel():5d}  max/median trung bình|x| = {(m.max() / m.median()).item():6.1f}")

In [ ]:
# explore: RTN vs GPTQ vs AWQ ở cùng cấu hình
methods = {
    "RTN": lambda n, W, X, b, gs, sym: rtn_quantize(W, b, gs, sym),
    "GPTQ": lambda n, W, X, b, gs, sym: gptq_quantize(W, hessian(X), b, gs, sym),
    "AWQ": lambda n, W, X, b, gs, sym: awq_quantize(W, X, b, gs, symmetric=sym)[0],
}
results = {}
for bits, gs, sym in [(4, 128, True), (3, 128, True), (3, 32, False)]:
    for label, fn in methods.items():
        start = time.perf_counter()
        q = apply_layerwise(model, inputs, lambda n, W, X: fn(n, W, X, bits, gs, sym))
        t_quant = time.perf_counter() - start
        results[(bits, gs, sym, label)] = val_bpc(q)
        print(f"int{bits} nhóm {gs} {'đối xứng' if sym else 'zero-point':10s} {label:5s} bpc {results[(bits, gs, sym, label)]:.4f}"
              f" (Δ {results[(bits, gs, sym, label)] - BASE_BPC:+.4f}, quantize {t_quant:.1f} s)")

In [ ]:
# explore: GPTQ tốt hơn ở đâu: sai số đầu ra từng lớp trên chính dữ liệu hiệu chỉnh, int3 nhóm 128
for n in names[:4]:
    W, X = model.get_submodule(n).weight.detach().cpu().float(), inputs[n]
    ref = X @ W.T
    e_rtn = ((X @ rtn_quantize(W, 3, 128).T - ref) ** 2).mean().item()
    e_gptq = ((X @ gptq_quantize(W, hessian(X), 3, 128).T - ref) ** 2).mean().item()
    e_w_rtn = ((rtn_quantize(W, 3, 128) - W) ** 2).mean().item()
    e_w_gptq = ((gptq_quantize(W, hessian(X), 3, 128) - W) ** 2).mean().item()
    print(f"{n:22s} sai số đầu ra RTN {e_rtn:.2e} GPTQ {e_gptq:.2e} | sai số trọng số RTN {e_w_rtn:.2e} GPTQ {e_w_gptq:.2e}")

## 6. Đọc kết quả

So với mục 5 phần lý thuyết. Những điều cần thấy:

1. GPTQ so với RTN ở từng cấu hình: khoảng cách lớn lên hay nhỏ đi khi bớt bit? Nối với bảng bpc của bài Intermediate 01.
2. Cell sai số từng lớp: GPTQ làm sai số **trọng số** to hơn hay nhỏ hơn RTN, còn sai số **đầu ra** thì sao? Đó chính là ý tưởng
   của GPTQ: tối ưu thứ ta quan tâm (đầu ra trên dữ liệu thật), không phải khoảng cách tới trọng số gốc.
3. AWQ: so với RTN và GPTQ, rồi đọc lại cell ngoại lai activation. Model nhỏ train ngắn có kênh activation lớn vượt trội như
   LLM tỉ tham số không? Mục 5 lý thuyết nói AWQ được thiết kế cho hiện tượng nào.
4. Thời gian quantize của mỗi phương pháp, và GPTQ/AWQ cần thứ gì mà RTN không cần.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.